In [ ]:
import duckdb
# path = "/home/bien/Documents/Development/RCS/robot-control-stack/utn_boxpnp_gripper"
path = "/home/bien/Documents/Development/RCS/robot-control-stack/examples/teleop/utn_usbc_insertion_with_absolute_action"
duckdb.sql(f"describe select *, from read_parquet('{path}')"), duckdb.sql(f"select count(*) as n_frames from read_parquet('{path}')")

In [ ]:
duckdb.sql(
f"""SELECT 
    info
FROM read_parquet('{path}')
WHERE success
limit 1
    """
)

In [ ]:
duckdb.sql(
    "SELECT count(DISTINCT uuid) as successful "
    f"FROM read_parquet('{path}') "
    "WHERE success"
),duckdb.sql(
    "SELECT count(DISTINCT uuid) as n_episodes "
    f"FROM read_parquet('{path}') "
)

In [ ]:
duckdb.sql(
f"""SELECT 
    MAX(step)/30, MIN(step)/30, AVG(step)/30
FROM read_parquet('{path}')
WHERE success
    """
)


In [ ]:
import io
import duckdb
from PIL import Image as PILImage
from IPython.display import Image, display

uuids = duckdb.sql(
    f"SELECT DISTINCT uuid FROM read_parquet('{path}') where SUCCESS"
).fetchnumpy()
episode = 2
n_frames = 400
frame_stride = 3


uuid1 = uuids["uuid"][episode]
rel = duckdb.read_parquet(path)

count = rel.filter(f"uuid='{uuid1}'").count("*").fetchone()[0]
info = rel.filter(f"uuid='{uuid1}'").select("info").fetchone()[0]
success = rel.filter(f"uuid='{uuid1}'").select("success").fetchone()[0]

print(f"episode uuid: {uuid1}, success: {success}, n_steps: {count}, info: {info}")

steps = list(range(0, n_frames * frame_stride, frame_stride))

frames = (
    rel
    .filter(f"uuid='{uuid1}' and step in ({','.join(map(str, steps))})")
    .select("step, obs.frames.digit_right_right.rgb.data AS img_data")
    .order("step")
    .fetchall()
)

print(f"Loaded {len(frames)} frames")

pil_frames = []

for step, img_data in frames:
    img = PILImage.open(io.BytesIO(img_data)).convert("RGB")
    pil_frames.append(img)

# Save to an in-memory GIF
gif_buffer = io.BytesIO()

pil_frames[0].save(
    gif_buffer,
    format="GIF",
    save_all=True,
    append_images=pil_frames[1:],
    duration=50,   # milliseconds per frame; 50 ms = 20 FPS
    loop=0,
)

gif_buffer.seek(0)

display(Image(data=gif_buffer.read(), format="gif"))

In [ ]:
import numpy as np
import pandas as pd
import duckdb
from matplotlib import pyplot as plt

episode = 13
uuids = duckdb.sql(
    f"SELECT DISTINCT uuid FROM read_parquet('{path}')"
).fetchnumpy()
episode = 0
n_frames = 400
frame_stride = 3
uuid1 = uuids["uuid"][episode]

df = duckdb.sql(f"""
    SELECT
        step,
        obs.right.tquat AS obs_tquat,
        action.right.tquat AS action_tquat,
        obs.right.gripper AS obs_gripper,
        action.right.gripper AS action_gripper,
    FROM read_parquet('{path}')
    WHERE uuid = '{uuid1}'
    ORDER BY step
""").df()

print(f"episode uuid: {uuid1}, n_steps: {len(df)}")

# Expand quaternion columns into x/y/z/w components
obs_tquat = np.stack(df["obs_tquat"][1:].to_numpy())
# print(df["action_tquat"].shape)
action_tquat = np.stack(df["action_tquat"][1:].to_numpy())
quat_names = ["x", "y", "z", "w"]
steps = df["step"][1:]

fig, ax = plt.subplots(1, 5, figsize=(26, 4), sharex=True)

for i, name in enumerate(quat_names):
    ax[i].plot(steps, obs_tquat[:, i], label="obs", linewidth=2)
    ax[i].plot(steps, action_tquat[:, i], label="action", linewidth=2, linestyle="--")
    ax[i].set_title(f"right.tquat.{name}")
    ax[i].set_xlabel("step")
    ax[i].grid(True, alpha=0.3)
    ax[i].legend()

ax[4].plot(steps, df["obs_gripper"][1:], label="obs", linewidth=2)
ax[4].plot(steps, df["action_gripper"][1:], label="action", linewidth=2, linestyle="--")
ax[4].set_title("right.gripper")
ax[4].set_xlabel("step")
ax[4].grid(True, alpha=0.3)
ax[4].legend()

plt.suptitle(f"UUID: {uuid1}", y=1.05)
plt.tight_layout()
plt.show()

# Save the <key> video from all episodes with stride as mp4

In [ ]:
import io
import math
from pathlib import Path

import duckdb
import imageio.v3 as iio
import numpy as np
from PIL import Image as PILImage


def export_tiled_episode_videos(
    path: str | Path,
    camera_cols: list[str],
    output_dir: str | Path = "episode_videos",
    fps: int = 20,
    frame_stride: int = 10,
    n_frames: int | None = None,
) -> None:
    """Export a tiled camera video for each successful episode.

    Existing videos are skipped.

    Output:
        output_dir/<uuid>/tiled.mp4

    Missing camera frames are rendered as black tiles.
    """
    if not camera_cols:
        raise ValueError("camera_cols must contain at least one camera column")

    path = Path(path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    generated_videos = []
    skipped_existing = 0
    con = duckdb.connect()

    try:
        uuids = con.execute(
            """
            SELECT DISTINCT uuid
            FROM read_parquet(?)
            WHERE success
            ORDER BY uuid
            """,
            [str(path)],
        ).fetchnumpy()["uuid"]

        print(f"Found {len(uuids)} successful episodes")

        limit_clause = f"LIMIT {n_frames}" if n_frames is not None else ""
        camera_selects = ",\n".join(
            f"{camera_col} AS camera_{index}"
            for index, camera_col in enumerate(camera_cols)
        )

        n_cameras = len(camera_cols)
        grid_cols = math.ceil(math.sqrt(n_cameras))
        grid_rows = math.ceil(n_cameras / grid_cols)

        for episode_idx, uuid in enumerate(uuids):
            episode_dir = output_dir / str(uuid)
            video_path = episode_dir / "tiled.mp4"

            if video_path.exists():
                skipped_existing += 1
                continue

            episode_dir.mkdir(parents=True, exist_ok=True)

            rows = con.execute(
                f"""
                SELECT
                    step,
                    {camera_selects}
                FROM read_parquet(?)
                WHERE uuid = ?
                  AND step % ? = 0
                ORDER BY step
                {limit_clause}
                """,
                [str(path), str(uuid), frame_stride],
            ).fetchall()

            if not rows:
                print(f"Skipping episode {episode_idx}: no frames found")
                continue

            tile_width = 0
            tile_height = 0

            for row in rows:
                for img_data in row[1:]:
                    if img_data is None:
                        continue

                    with PILImage.open(io.BytesIO(img_data)) as img:
                        tile_width = max(tile_width, img.width)
                        tile_height = max(tile_height, img.height)

            if tile_width == 0 or tile_height == 0:
                print(f"Skipping episode {episode_idx}: all camera frames were empty")
                continue

            frames = []

            for row in rows:
                tiled_frame = np.zeros(
                    (grid_rows * tile_height, grid_cols * tile_width, 3),
                    dtype=np.uint8,
                )

                for camera_idx, img_data in enumerate(row[1:]):
                    if img_data is None:
                        continue

                    with PILImage.open(io.BytesIO(img_data)) as img:
                        img = img.convert("RGB")
                        img.thumbnail((tile_width, tile_height))
                        img_array = np.asarray(img)

                    tile_row = camera_idx // grid_cols
                    tile_col = camera_idx % grid_cols
                    y = tile_row * tile_height
                    x = tile_col * tile_width

                    y_offset = (tile_height - img_array.shape[0]) // 2
                    x_offset = (tile_width - img_array.shape[1]) // 2

                    tiled_frame[
                        y + y_offset : y + y_offset + img_array.shape[0],
                        x + x_offset : x + x_offset + img_array.shape[1],
                    ] = img_array

                frames.append(tiled_frame)

            iio.imwrite(
                video_path,
                frames,
                fps=fps,
                codec="libx264",
                pixelformat="yuv420p",
            )

            generated_videos.append(video_path)

        print(f"\nGenerated {len(generated_videos)} new video(s):")

        for video_path in generated_videos:
            print(f"  {video_path}")

        print(f"\nSkipped {skipped_existing} existing video(s).")
        print("Done.")

    finally:
        con.close()

In [ ]:
export_tiled_episode_videos(
    path=path,
    camera_cols=[
        "obs.frames.side.rgb.data",
        "obs.frames.wrist.rgb.data",
        "obs.frames.digit_right_left.rgb.data",
        "obs.frames.digit_right_left_blank.rgb.data",
        "obs.frames.digit_right_right.rgb.data",
        "obs.frames.digit_right_right_blank.rgb.data",
    ],
)

# DANGER ZONE: DELETE EPISODE

In [ ]:
import duckdb
import shutil
from pathlib import Path
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq

path = Path(
    "/home/bien/Documents/Development/RCS/robot-control-stack/"
    "examples/teleop/utn_usbc_insertion"
)
# Removed uuids from original utn_usbc_insertion dataset
# "ebebe18d4df249f7b59ff5ae18df81e1",
# "fbb2cd6390624d89a07a9c01817ebc3a",
# "7caf079e73d045eb93a38988522522e9",
# "74e83fbe08a24d56bd000ff5712e9237",
# "216ced430c43437db9864dc4709f446e",
# "264ef8ca2c684704964dff423a4790d2",
# "58754f76f81c47f8be52ed93c16596ea",
# "462514e9e5934ed6b72911e237aa9353",
# "6d7227dc64a94c029cd0b81ce61297f0",
# "d8110c285e09443ca239afda30b32adc"

uuids_to_remove = [
]


if not uuids_to_remove:
    raise ValueError("uuids_to_remove cannot be empty.")

if not path.is_dir():
    raise ValueError("This version expects `path` to be a dataset directory.")

temp_path = path.with_name(path.name + "_filtered_tmp")
backup_path = path.with_name(path.name + "_backup")
validation_path = path.with_name(path.name + "_consolidation_validation_tmp")

if temp_path.exists():
    shutil.rmtree(temp_path)

if backup_path.exists():
    raise FileExistsError(f"Backup already exists: {backup_path}")

if validation_path.exists():
    shutil.rmtree(validation_path)

source_files = sorted(path.rglob("*.parquet"))
if not source_files:
    raise FileNotFoundError(f"No Parquet files found in {path}")

bad_uuid_values = pa.array(uuids_to_remove, type=pa.string())


def count_rows_and_matches(files: list[Path]) -> tuple[int, int]:
    """Count total rows and rows whose UUID is scheduled for removal."""
    total_rows = 0
    matching_rows = 0

    for parquet_file in files:
        parquet = pq.ParquetFile(parquet_file)

        if "uuid" not in parquet.schema_arrow.names:
            raise ValueError(f"Missing 'uuid' column: {parquet_file}")

        for batch in parquet.iter_batches(columns=["uuid"]):
            uuid_column = batch.column(0)
            matches = pc.is_in(uuid_column, value_set=bad_uuid_values)

            total_rows += batch.num_rows
            matching_rows += pc.sum(matches).as_py() or 0

    return total_rows, matching_rows


def filter_parquet_file(source_file: Path, destination_file: Path) -> int:
    """Filter one Parquet fragment while preserving its Arrow schema."""
    source_parquet = pq.ParquetFile(source_file)
    source_schema = source_parquet.schema_arrow
    uuid_index = source_schema.get_field_index("uuid")

    if uuid_index == -1:
        raise ValueError(f"Missing 'uuid' column: {source_file}")

    destination_file.parent.mkdir(parents=True, exist_ok=True)
    removed_rows = 0

    # The schema is taken directly from the original fragment, rather than
    # inferred through DuckDB. This preserves nested observation/image fields.
    with pq.ParquetWriter(
        destination_file,
        source_schema,
        compression="zstd",
    ) as writer:
        # Do not use ``iter_batches`` here.  It asks Parquet to produce
        # RecordBatches directly, which PyArrow cannot do for some nested
        # schemas (``ArrowNotImplementedError: Nested data conversions not
        # implemented for chunked array outputs``).  Reading one row group
        # produces an Arrow Table instead, whose ChunkedArrays support those
        # nested fields.
        for row_group_index in range(source_parquet.num_row_groups):
            row_group = source_parquet.read_row_group(row_group_index)
            uuid_column = row_group.column(uuid_index)

            # Retain null UUIDs and any UUID not listed for removal.
            keep_mask = pc.or_(
                pc.is_null(uuid_column),
                pc.invert(pc.is_in(uuid_column, value_set=bad_uuid_values)),
            )

            filtered_row_group = row_group.filter(keep_mask)
            removed_rows += row_group.num_rows - filtered_row_group.num_rows

            if filtered_row_group.num_rows:
                writer.write_table(filtered_row_group)

    # Confirm the rewritten fragment retained precisely the original schema.
    output_schema = pq.ParquetFile(destination_file).schema_arrow
    if not output_schema.equals(source_schema, check_metadata=True):
        raise RuntimeError(
            f"Schema changed while filtering {source_file.name}. "
            "Original data has not been modified."
        )

    return removed_rows


original_count, matching_rows = count_rows_and_matches(source_files)
print(f"Rows matching listed UUIDs: {matching_rows}")

if matching_rows == 0:
    print("No matching UUIDs found. Nothing changed.")

else:
    try:
        # Retain every non-Parquet file and all unaffected Parquet fragments.
        shutil.copytree(path, temp_path)

        rewritten_files = []
        removed_rows = 0

        for source_file in source_files:
            parquet = pq.ParquetFile(source_file)

            fragment_matches = 0
            for batch in parquet.iter_batches(columns=["uuid"]):
                fragment_matches += (
                    pc.sum(
                        pc.is_in(
                            batch.column(0),
                            value_set=bad_uuid_values,
                        )
                    ).as_py()
                    or 0
                )

            # Do not touch fragments which contain no UUIDs being removed.
            if fragment_matches == 0:
                continue

            relative_file = source_file.relative_to(path)
            replacement_file = temp_path / relative_file

            removed_rows += filter_parquet_file(
                source_file,
                replacement_file,
            )
            rewritten_files.append(relative_file)

        output_files = sorted(temp_path.rglob("*.parquet"))
        filtered_count, remaining_matches = count_rows_and_matches(output_files)

        if remaining_matches != 0:
            raise RuntimeError(
                f"Verification failed: {remaining_matches} matching rows remain."
            )

        if original_count - filtered_count != matching_rows:
            raise RuntimeError(
                "Verification failed: removed-row count does not match "
                "the number of matching source rows."
            )

        # Validate using the same dataset write operation used by RCS
        # consolidation. This occurs in a disposable directory; the original
        # dataset has not been changed at this point.
        part_scheme = ds.partitioning(
            schema=pa.schema([pa.field("date", pa.string())]),
            flavor="filename",
        )

        validation_dataset = ds.dataset(
            temp_path,
            format="parquet",
            partitioning=part_scheme,
        )

        ds.write_dataset(
            data=validation_dataset,
            base_dir=validation_path,
            format="parquet",
            partitioning=part_scheme,
            existing_data_behavior="overwrite_or_ignore",
        )

        # The validation write succeeded, so discard its disposable output.
        shutil.rmtree(validation_path)

        # Only now replace the original dataset and retain its backup.
        path.rename(backup_path)
        temp_path.rename(path)

        print(f"Removed rows:   {removed_rows}")
        print(f"Remaining rows: {filtered_count}")
        print(f"Rewritten fragments: {len(rewritten_files)}")

        for file_path in rewritten_files:
            print(f"  {file_path}")

        print(f"Updated data:   {path}")
        print(f"Backup:         {backup_path}")

    except Exception:
        # Leave the original untouched and remove incomplete temporary results.
        if temp_path.exists():
            shutil.rmtree(temp_path)

        if validation_path.exists():
            shutil.rmtree(validation_path)

        raise
